# Simulation Code T = 3

## Import libraries

In [14]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.dynamicRieszBradic import estimateBradic
import torch
import pandas as pd
import time

## Estimation Settings

In [15]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # To speed up
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # To speed up
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

rf_f_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

#### Simulation settings:

In [16]:
torch.manual_seed(123);

# Parameters
N = 1000
tmax = 1

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = dimX3 = 3
dimX1 = 3
dimX2 = 3
dimX3 = 3

#### Define treatment probability function

In [17]:
# Bounds (only for truncated distributions)
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Define func_link function (nonlinear probability function from Bradic)
def func_link(x):
    return (x > 0) * torch.abs(x) / (torch.abs(x) + 1) + (x < 0) / (torch.abs(x) + 1)

# Define a truncated func_link function
def truncated_link(x):
    return lower + (upper - lower) * func_link(x)

# Simple highly nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

In [18]:
treatment_probability_func = truncated_logistic

#### Generate data

In [19]:
# Coefficients
beta_g0_0 = 1
beta_g1_0 = -1

beta_g0_S1 = torch.tensor([1, 1, -1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a' first part (paper)
beta_g1_S1 = torch.tensor([-1, 1, -1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a first part (paper)

beta_g0_S2 = torch.tensor([1, 1, 1] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a' second part (paper)
beta_g1_S2 = torch.tensor([-1, -1, 1] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a second part (paper)

beta_pi1_0 = 0
beta_pi1_S1 = torch.tensor([1, 1, 1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # gamma_a i(paper)

beta_pi2_0_0 = 0
beta_pi2_0_S1 = torch.tensor([0.5, 0, -0.5] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' first part (paper)
beta_pi2_0_S2 = torch.tensor([0.5, 0, 0.5] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' second part (paper)

beta_pi2_1_0 = 0
beta_pi2_1_S1 = torch.tensor([1, 1, 0] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a first part (paper)
beta_pi2_1_S2 = torch.tensor([1, -1, 0] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)

# Now extend to T = 3
beta_g0_S3 = torch.tensor([-1, 1, 1] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a' second part (paper)
beta_g1_S3 = torch.tensor([1, -1, -1] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a second part (paper)

beta_pi3_00_0 = 0
beta_pi3_00_S1 = torch.tensor([0.5, 0, -0.5] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' first part (paper)
beta_pi3_00_S2 = torch.tensor([0.5, 0, 0.5] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' second part (paper)
beta_pi3_00_S3 = torch.tensor([-0.5, 0.5, -0.5] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' second part (paper)

beta_pi3_11_0 = 0
beta_pi3_11_S1 = torch.tensor([1, 1, 0] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a first part (paper)
beta_pi3_11_S2 = torch.tensor([1, -1, 0] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)
beta_pi3_11_S3 = torch.tensor([-1, 1, -1] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)

beta_pi3_01_0 = 0
beta_pi3_01_S1 = torch.tensor([1, 1, 1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a first part (paper)
beta_pi3_01_S2 = torch.tensor([1, -1, 1] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)
beta_pi3_01_S3 = torch.tensor([-1, 1, 1] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)

beta_pi3_10_0 = 0
beta_pi3_10_S1 = torch.tensor([0, 1, 0] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a first part (paper)
beta_pi3_10_S2 = torch.tensor([0, -1, 0] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)
beta_pi3_10_S3 = torch.tensor([0, 1, -1] + [0] * (dimX3 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)

# Generate data

# Period 1
S1_all = torch.randn(N * tmax, dimX1)
pi1_all = treatment_probability_func(beta_pi1_0 + S1_all @ beta_pi1_S1).reshape(-1,1)
A1_all = torch.bernoulli(pi1_all).int().reshape(-1,1)

# Period 2
delta1_all = torch.randn(N * tmax,1)
delta2_all = torch.randn(N * tmax, dimX2)

S2_all = S1_all + A1_all * (1 + delta1_all) + delta2_all
S2_1_all = S1_all + 1 + delta1_all + delta2_all
S2_0_all = S1_all + delta2_all

pi2_0_all = treatment_probability_func(beta_pi2_0_0 + S1_all @ beta_pi2_0_S1 + S2_0_all @ beta_pi2_0_S2)
pi2_1_all = treatment_probability_func(beta_pi2_1_0 + S1_all @ beta_pi2_1_S1 + S2_1_all @ beta_pi2_1_S2)
pi2_all = treatment_probability_func((1 - A1_all) * (beta_pi2_0_0 + S1_all @ beta_pi2_0_S1 + S2_0_all @ beta_pi2_0_S2) + 
                   A1_all * (beta_pi2_1_0 + S1_all @ beta_pi2_1_S1 + S2_1_all @ beta_pi2_1_S2))

A2_all = torch.bernoulli(pi2_all).int() 

# Period 3
delta3_all = torch.randn(N * tmax, dimX3)
delta3_2_all = torch.randn(N * tmax, dimX3)

S3_all = S2_all + A2_all * (1 + delta3_2_all) + delta3_all
S3_11_all = S2_1_all + (1 + delta3_2_all ) + delta3_all
S3_10_all = S2_1_all + delta3_all
S3_01_all = S2_0_all + 1 + delta3_2_all + delta3_all
S3_00_all = S2_0_all + delta3_all

pi3_11_all = treatment_probability_func(beta_pi3_11_0 + S1_all @ beta_pi3_11_S1 + S2_all @ beta_pi3_11_S2 + S3_all @ beta_pi3_11_S3)
pi3_10_all = treatment_probability_func(beta_pi3_10_0 + S1_all @ beta_pi3_10_S1 + S2_all @ beta_pi3_10_S2 + S3_all @ beta_pi3_10_S3)
pi3_01_all = treatment_probability_func(beta_pi3_01_0 + S1_all @ beta_pi3_01_S1 + S2_all @ beta_pi3_01_S2 + S3_all @ beta_pi3_01_S3)
pi3_00_all = treatment_probability_func(beta_pi3_00_0 + S1_all @ beta_pi3_00_S1 + S2_all @ beta_pi3_00_S2 + S3_all @ beta_pi3_00_S3)

pi3_all = ((A1_all + A2_all == 0).float() * pi3_00_all + 
         (A1_all * A2_all == 1).float() * pi3_11_all + 
         (A1_all * ( 1 - A2_all ) == 1).float() * pi3_10_all + 
         (( 1 - A1_all ) * A2_all == 1).float() * pi3_01_all)
A3_all = torch.bernoulli(pi3_all).int()

# Outcome
g_all = ((A1_all + A2_all + A3_all == 0).float() * (beta_g0_0 + S1_all @ beta_g0_S1 + S2_all @ beta_g0_S2 + S3_all @ beta_g0_S3) + 
         (A1_all * A2_all * A3_all == 1).float() * (beta_g1_0 + S1_all @ beta_g1_S1 + S2_all @ beta_g1_S2 + S3_all @ beta_g1_S3))
zeta_all = torch.randn(N * tmax,1)
Y_all = g_all + zeta_all

#### True values:

In [20]:
mu3_1_all = beta_g1_0 + S1_all @ beta_g1_S1 + S2_1_all @ beta_g1_S2 + S3_11_all @ beta_g1_S3
mu3_0_all = beta_g0_0 + S1_all @ beta_g0_S1 + S2_0_all @ beta_g0_S2 + S3_00_all @ beta_g0_S3

mu2_1_all = beta_g1_0 + S1_all @ beta_g1_S1 + S2_1_all @ (beta_g1_S2 + beta_g1_S3) + beta_g1_S3.sum()
mu2_0_all = beta_g0_0 + S1_all @ beta_g0_S1 + S2_0_all @ (beta_g0_S2 + beta_g0_S3)

mu1_1_all = beta_g1_0 + S1_all @ (beta_g1_S1 + beta_g1_S2 + beta_g1_S3) + beta_g1_S2.sum() + 2 * beta_g1_S3.sum()
mu1_0_all = beta_g0_0 + S1_all @ (beta_g0_S1 + beta_g0_S2 + beta_g0_S3)

theta0 = beta_g0_0 
theta1 = beta_g1_0 + beta_g1_S2.sum() + 2 * beta_g1_S3.sum()
theta = theta1 - theta0
print(theta, theta1, theta0)

tensor(-5.) tensor(-4.) 1


## Estimation:

#### Estimation settings

In [21]:
folds = 2

time0 = time.time()

#### Estimation 

In [ ]:
pred_theta = torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)

for t in range(0,tmax):
    # Get data for current iteration
    S1, S2, S3 = S1_all[(t) * N : (t+1) * N, :], S2_all[(t) * N : (t+1) * N, :], S3_all[(t) * N : (t+1) * N, :]
    A1, A2, A3 = A1_all[(t) * N : (t+1) * N], A2_all[(t) * N : (t+1) * N], A3_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = torch.hstack((S1,S2,S3))
    X_index = torch.tensor([S1.shape[1]-1,S1.shape[1]+S2.shape[1]-1, S1.shape[1]+S2.shape[1] + S3.shape[1]-1], dtype=torch.int32).reshape(-1,1)
    D = torch.hstack((A1,A2,A3))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    pi3_11, pi3_10, pi3_01, pi3_00 = pi3_11_all[(t) * N : (t+1) * N], pi3_10_all[(t) * N : (t+1) * N], pi3_01_all[(t) * N : (t+1) * N], pi3_00_all[(t) * N : (t+1) * N]
    mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    mu3_1, mu3_0 = mu3_1_all[(t) * N : (t+1) * N], mu3_0_all[(t) * N : (t+1) * N]

    if d[0,0] == 1:
        gamma3_1,gamma2_1,gamma1_1  = A1 * A2 * A3 / (pi1 * pi2_1 * pi3_11), A1 * A2 / (pi1 * pi2_1), A1 / pi1 
        theta_hat = (gamma3_1 * Y - (gamma1_1 - 1) * mu1_1  - (gamma2_1 - gamma1_1) * mu2_1 - (gamma3_1 - gamma2_1) * mu3_1)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    elif d[0,0] == 0:
        gamma3_0, gamma2_0, gamma1_0 = (1 - A1) * (1 - A2) * (1 - A3) / ((1 - pi1  ) * (1 - pi2_0  ) *  (1 - pi3_00  )), (1 - A1) * (1 - A2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - A1) / (1 - pi1)
        theta_hat = (gamma3_0 * Y - (gamma1_0 - 1) * mu1_0   - (gamma2_0 - gamma1_0) * mu2_0 - (gamma3_0 - gamma2_0) * mu3_0)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    #---------------------------------------------------------------------------
    # Estimator 2: Bradic
    # Currently not implemented

    #---------------------------------------------------------------------------
    # Estimator 3: LASSO 

    pred_theta[t,2], pred_sig[t,2], *_ = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
                                                                     method_a = "LASSO", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "LASSO", lasso_f_settings = lasso_f_settings)

    #---------------------------------------------------------------------------
    # Estimator 4: RF 

    pred_theta[t,3], pred_sig[t,3], *_ = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
                                                                     method_a = "RF", rf_a_settings = rf_a_settings,
                                                                        method_f = "RF", rf_f_settings = rf_f_settings)

    #---------------------------------------------------------------------------
    # Estimator 5: Net 

    # pred_theta[t,4], pred_sig[t,4], *_ = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
    #                                                                  method_a = "Net", net_a_settings = net_a_settings,
    #                                                                     method_f = "Net", net_f_settings = net_f_settings)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.4124157726764679, tolerance: 0.3288384675979614
  model = cd_fast.enet_coordinate_descent_gram(
/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.37977516651153564, tolerance: 0.3288384675979614
  model = cd_fast.enet_coordinate_descent_gram(
/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.3796069920063019, tolerance: 0.3288384675979614
  model = cd_fast.enet_coordinate_descent_gram(
/Users/wisserutgers/anaconda3/lib/pytho

Time until iteration  0 :  37.0656099319458
Finished. Total time:  37.065813064575195


## Compute statistics

#### Get true value

In [23]:
true_value = theta1

#### Compute statistics

In [24]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [25]:
df = pd.DataFrame(data)
print(df)

   Method      Bias      RMSE  Coverage  Interval Length
0  Oracle  0.222070  0.222070       1.0         1.482388
1  Bradic  4.000000  4.000000       0.0         0.000000
2   LASSO  0.319185  0.319185       1.0         1.593003
3      RF  0.524713  0.524713       0.0         0.916189
4     Net  4.000000  4.000000       0.0         0.000000


In [27]:
pred_theta

tensor([[-3.7779,  0.0000, -3.6808, -3.4753,  0.0000]])

In [26]:
# check_t = 0
# Compare RR:
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
